In [ ]:
from pyspark.sql import functions as F
 
# --------------------------------------------
# 0) Source tables (Unity Catalog)
# --------------------------------------------
sales_src   = spark.table("genai_demo.altreyx_demo.tableupdated_new_1000")
channel_src = spark.table("genai_demo.altreyx_demo.tdmedpod_new_1")
 
# Helper: COALESCE(x, 0)
def nz(colname):
    return F.coalesce(F.col(colname), F.lit(0.0))
 
# Ensure we treat dates correctly even if stored as strings
so_audat_col = F.to_date(F.col("so_audat"))
fkdat_col    = F.to_date(F.col("fkdat"))
 
# --------------------------------------------
# 1) Build run-time dates FROM DATA (min/max in sales table)
#    'today' := max(so_audat)
# --------------------------------------------
bounds = sales_src.select(
    F.max(so_audat_col).alias("max_date"),
    F.min(so_audat_col).alias("min_date"),
)
 
# Single-row date scaffold, aligned to your original fields but
# computed relative to max_date instead of current_date()
dc = (
    bounds.select(
        F.col("max_date").alias("today"),
        F.date_sub(F.col("max_date"), 1).alias("yesterday"),
 
        # Prior week start/end relative to 'today'
        F.date_sub(F.date_trunc("week", F.col("max_date")), 7).cast("date").alias("prior_week_start"),
        F.date_sub(F.date_trunc("week", F.col("max_date")), 1).cast("date").alias("prior_week_end"),
 
        # Month windows relative to 'today'
        F.add_months(F.trunc(F.col("max_date"), "MONTH"), -1).alias("p1m_start"),
        F.date_sub(F.col("max_date"), 1).alias("p1m_end"),
        F.add_months(F.trunc(F.col("max_date"), "MONTH"), -2).alias("p2m_start"),
        F.date_sub(F.add_months(F.trunc(F.col("max_date"), "MONTH"), -1), 1).alias("p2m_end"),
        F.add_months(F.trunc(F.col("max_date"), "MONTH"), -3).alias("p3m_start"),
        F.date_sub(F.add_months(F.trunc(F.col("max_date"), "MONTH"), -2), 1).alias("p3m_end"),
 
        # Sliding windows (kept for parity)
        F.date_sub(F.col("max_date"), 14).alias("start_2wk"),
        F.date_sub(F.col("max_date"), 7).alias("end_2wk"),
        F.date_sub(F.col("max_date"), 28).alias("start_4wk"),
        F.date_sub(F.col("max_date"), 21).alias("end_4wk"),
        F.date_sub(F.col("max_date"), 35).alias("start_5wk"),
        F.date_sub(F.col("max_date"), 28).alias("end_5wk"),
        # Also expose min/max if you want to use full range
        F.col("min_date").alias("min_date"),
        F.col("max_date").alias("max_date_dup")  # same as today
    )
)
 
# --------------------------------------------
# 2) SALES DATA (same CTE logic; filter uses data-driven dates)
# --------------------------------------------
sales_data = (
    sales_src.alias("s")
    .crossJoin(dc)
    .where(F.to_date(F.col("s.so_audat")).between(F.col("p2m_end"), F.col("today")))
    # If you want FULL available range instead of p2m_end→today, use:
    # .where(F.to_date(F.col("s.so_audat")).between(F.col("min_date"), F.col("max_date_dup")))
    .groupBy(
        F.col("s.so_audat").alias("SO_Date"),
        F.col("s.fkdat").alias("BILL_DATE"),
        F.col("s.werks").alias("Whs"),
        F.col("s.vtweg").cast("string").alias("DIST_CHNL_ID"),
        F.col("s.zzfinclass").alias("FNC_ID"),
        F.col("s.bezek").alias("FNC_DESC"),
        F.col("s.soldto_kunnr").alias("SOLDTO"),
        F.col("s.shipto_kunnr").alias("SHIPTO"),
        F.col("s.vgbel").alias("RFRNC_DOC_NUM"),
    )
    .agg(
        F.count_distinct("s.vgbel").alias("Invoices"),
        F.sum(nz("s.bill_itm_count")).alias("Invoice_Lines"),
        F.sum(nz("s.so_netwr")).alias("SO_NetValue_Amt"),
        F.sum(nz("s.so_netpr")).alias("SO_NetPrice_Amt"),
        F.sum(nz("s.fkimg")).alias("SELL_QTY"),
        F.sum(nz("s.fklmg")).alias("BASE_QTY"),
        F.sum(nz("s.vbrp_brgew")).alias("WGT"),
        F.sum(nz("s.vbrp_volum")).alias("VOL"),
        F.sum(nz("s.extnd_land_cst")).alias("LANDED_COST"),
        F.sum(nz("s.extnd_fnl_price")).alias("EXT_FINAL_PRICE"),
        F.sum(nz("s.extnd_fnl_price")).alias("Invoice_Sales"),  # override per SQL
        F.sum(nz("s.addtn_trans_fee_ovrride_zsro")).alias("Rush_Order_Fee"),
        F.sum(nz("s.rf_trnsct_absorb_charge_amt_ztr2")).alias("Trans_Absorb_Amt"),
        F.sum(nz("s.rf_trnsct_charge_amt_ztrm")).alias("Trans_Charge_Amt"),
        F.sum(nz("s.vndr_hndlng_amt_zthm")).alias("Vendor_Hndl_Amt"),
        F.sum(nz("s.rf_min_order_charge_amt_zsmo")).alias("MOC_Amt"),
        F.sum(nz("s.fuel_surcharge_zsdf")).alias("Fuel_Surcharge"),
        F.sum(nz("s.markup_vendor_trans_fee_amt_zmt1")).alias("Markup_Vendor_Trans"),
    )
)
 
# --------------------------------------------
# 3) CHANNEL DATA (same as your SQL CTE)
# --------------------------------------------
channel_data = (
    channel_src
    .where((F.col("bill_dte") == F.lit("2019-07-01")) & (F.col("whs") == F.lit("D0CG")))
    .select(
        F.col("dist_chnl_id").cast("string").alias("dist_chnl_id"),
        F.col("dist_chnl_desc").alias("dist_chnl_desc"),
        F.col("direct_std_cost").alias("direct_std_cost"),
        F.col("net_rev_amt").alias("net_rev_amt"),
        F.col("rev_cost").alias("rev_cost"),
    )
)
 
# --------------------------------------------
# 4) FINAL JOIN (same as your final_data CTE)
# --------------------------------------------
final_data = (
    sales_data.alias("sd")
    .join(channel_data.alias("cd"),
          F.col("sd.DIST_CHNL_ID") == F.col("cd.dist_chnl_id"),
          "left")
    .select(
        "sd.SO_Date",
        "sd.BILL_DATE",
        "sd.Whs",
        "sd.DIST_CHNL_ID",
        "cd.dist_chnl_desc",
        "sd.FNC_ID",
        "sd.FNC_DESC",
        "sd.SOLDTO",
        "sd.SHIPTO",
        "sd.RFRNC_DOC_NUM",
        "sd.Invoices",
        "sd.Invoice_Lines",
        "sd.Invoice_Sales",
        F.col("sd.EXT_FINAL_PRICE").alias("ext_sales"),
        "sd.Rush_Order_Fee",
        (F.col("sd.Trans_Charge_Amt") + F.col("sd.Vendor_Hndl_Amt") + F.col("sd.MOC_Amt") + F.col("sd.Fuel_Surcharge")).alias("BIA_Ship_Hndl_Amt"),
        (F.col("sd.Trans_Absorb_Amt") + F.col("sd.Markup_Vendor_Trans")).alias("COE_Ship_Hndl_Amt"),
    )
)
 
# --------------------------------------------
# 5) USE RESULT
# --------------------------------------------
final_data.show(50, truncate=False)
 
final_data.count()
 
# Optional persist:
# final_data.write.mode("overwrite").saveAsTable("genai_demo.altreyx_demo.final_data_table")
